# Stage 1: Dataset Analysis & Leakage Investigation
**AI/ML Capstone — Phase 1: Machine Learning Pipeline**  
**Dataset:** Water Quality Monitoring (`data/raw/Combined_dataset.csv`, 2,827,977 rows × 14 columns)  
**Objective:** Comprehensive read-only audit of data properties, physical plausibility, CCME WQI target structure, and formula leakage risk.

---

### Key Stage 1 Milestones:
1. **Data Ingestion & Parquet Caching**: Load 2.8M rows with explicit data types and datetime parsing; cache to Parquet for fast reproducible access.
2. **Structural & Memory Diagnostics**: Assess memory footprint, null counts, duplicate records, and distinct cardinality.
3. **Distribution Profiling**: Compute summary percentiles for 8 chemistry variables and the target (`CCME_Values`).
4. **Group Distributions**: Breakdown by Country, Waterbody Type, Area monitoring density, and temporal span (1940–2023).
5. **Physical Plausibility & Sensor Errors**: Flag non-physical pH (<0 or >14), extreme temperatures (>40°C), negative concentrations, and zero DO.
6. **Reverse-Engineering the CCME WQI Target**: Empirically derive the exact decision boundaries between CCME continuous values and 5 discrete classes (`Excellent`, `Good`, `Fair`, `Marginal`, `Poor`).
7. **Leakage & Formula Analysis**: Quantify the risk of "re-learning an arithmetic formula" ($WQI = 100 - \frac{\sqrt{F_1^2 + F_2^2 + F_3^2}}{1.732}$) from exceedances.

In [1]:
%load_ext autoreload
%autoreload 2

import os
import sys
import json
from pathlib import Path
from collections import OrderedDict
import numpy as np
import pandas as pd
from tabulate import tabulate

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

SEED = 42
np.random.seed(SEED)
pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 1000)

print(f"Working Project Root: {PROJECT_ROOT}")

Working Project Root: /Users/musthyalasadhvik/Desktop/projects/aml-capstone/AML_Capstone


## 1. Data Ingestion & Parquet Caching
We load the raw CSV (`data/raw/Combined_dataset.csv`) with explicit dtypes (`category`, `float64`, parsed dates) and cache it to Parquet (`data/processed/raw_cached.parquet`) for fast I/O and zero modification of raw files.

In [2]:
from src.stage1_analysis import load_data, CHEMISTRY_COLS, TARGET_CONT, TARGET_CLASS, META_COLS

df = load_data()
print(f"Loaded dataset: {df.shape[0]:,} rows × {df.shape[1]} columns")
print("First 5 rows:")
print(tabulate(df.head(), headers="keys", tablefmt="github", showindex=False))

Loading cached parquet: /Users/musthyalasadhvik/Desktop/projects/aml-capstone/AML_Capstone/data/processed/raw_cached.parquet
Loaded dataset: 2,827,977 rows × 14 columns
First 5 rows:
| Country   | Area            | Waterbody Type   | Date                |   Ammonia (mg/l) |   Biochemical Oxygen Demand (mg/l) |   Dissolved Oxygen (mg/l) |   Orthophosphate (mg/l) |   pH (ph units) |   Temperature (cel) |   Nitrogen (mg/l) |   Nitrate (mg/l) |   CCME_Values | CCME_WQI   |
|-----------|-----------------|------------------|---------------------|------------------|------------------------------------|---------------------------|-------------------------|-----------------|---------------------|-------------------|------------------|---------------|------------|
| Canada    | SE649035-145565 | River            | 1974-01-12 00:00:00 |        0.059248  |                               1.3  |                    8.15   |              0.0119167  |         8.075   |               9.885 |          0.3

## 2. Basic Information & Data Integrity
Inspect dataset dimensions, memory footprint, data types, missing values (NaNs), exact duplicates, and distinct cardinality per column.

In [3]:
from src.stage1_analysis import basic_info

info = basic_info(df)

print(f"Shape: {info['shape'][0]:,} rows × {info['shape'][1]} columns")
print(f"Memory footprint: {info['memory_mb']:.2f} MB")
print(f"Total missing values: {info['total_missing']}")
print(f"Exact duplicate rows: {info['exact_duplicates']:,}")

unique_df = pd.DataFrame([
    {"Column": col, "Dtype": str(df[col].dtype), "Unique Values": f"{info['unique_counts'][col]:,}"}
    for col in df.columns
])
print("Column Summary:")
print(tabulate(unique_df, headers="keys", tablefmt="github", showindex=False))

Shape: 2,827,977 rows × 14 columns
Memory footprint: 328.49 MB
Total missing values: 0
Exact duplicate rows: 9,971
Column Summary:
| Column                           | Dtype          |   Unique Values |
|----------------------------------|----------------|-----------------|
| Country                          | category       |               5 |
| Area                             | str            |          61,689 |
| Waterbody Type                   | category       |              12 |
| Date                             | datetime64[us] |          23,876 |
| Ammonia (mg/l)                   | float64        |          13,987 |
| Biochemical Oxygen Demand (mg/l) | float64        |           4,717 |
| Dissolved Oxygen (mg/l)          | float64        |          16,253 |
| Orthophosphate (mg/l)            | float64        |          10,596 |
| pH (ph units)                    | float64        |           4,726 |
| Temperature (cel)                | float64        |          11,536 |
| Nit

## 3. Numeric Summary & Percentiles
We compute percentiles across the 8 water chemistry parameters and the continuous target (`CCME_Values`). Notice the minimum values, maximum values, and the ceiling effect at 100.0 for `CCME_Values`.

In [4]:
from src.stage1_analysis import percentile_summary

pct_df = percentile_summary(df)
print(tabulate(pct_df.round(4), headers="keys", tablefmt="github"))

|                                  |       count |    mean |     std |     min |      0% |      1% |      5% |     10% |     25% |     50% |     75% |      90% |      95% |    99% |   100% |   max |
|----------------------------------|-------------|---------|---------|---------|---------|---------|---------|---------|---------|---------|---------|----------|----------|--------|--------|-------|
| Ammonia (mg/l)                   | 2.82798e+06 |  1.1714 |  5.6689 | -0.005  | -0.005  |  0.009  |  0.021  |  0.021  |  0.03   |  0.055  |   0.317 |   1.54   |   4.4582 |  26.7  |    200 |   200 |
| Biochemical Oxygen Demand (mg/l) | 2.82798e+06 |  4.8866 | 16.4136 | -2      | -2      |  0.7    |  1      |  1.1    |  1.6    |  2.7    |   2.83  |   6.85   |  11.7    |  68.2  |    255 |   255 |
| Dissolved Oxygen (mg/l)          | 2.82798e+06 | 10.008  |  1.8511 |  0      |  0      |  3.5505 |  5.7    |  8.1    |  9.86   | 10.2    |  11     |  11.6841 |  12.2    |  13.5  |     20 |    20 |
| Ort

## 4. Metadata Group Distributions
Examine distribution shifts and geographic / environmental coverage:
- **Country Representation**: England constitutes ~75.3% of the samples, followed by USA (14.6%), Ireland (8.3%), China (1.6%), and Canada (0.1%).
- **Waterbody Representation**: Dominance of Rivers (65.5%), followed by Effluent (21.3%) and Lakes (5.4%). Effluent and Sewage represent non-ambient sources.
- **Temporal Distribution**: Spanning 1940 to 2023, with >90% of samples clustered between 2009 and 2023.
- **Monitoring Sites (Areas)**: 61,689 distinct monitoring sites with highly skewed observation counts.

In [5]:
import importlib
import src.stage1_analysis
importlib.reload(src.stage1_analysis)
from src.stage1_analysis import group_counts

groups = group_counts(df)

print("### By Country:")
print(tabulate(groups["country_df"], headers="keys", tablefmt="github", showindex=False))

print("\n### By Waterbody Type:")
print(tabulate(groups["waterbody_df"], headers="keys", tablefmt="github", showindex=False))

print(f"\nTemporal Span: {groups['year_min']} – {groups['year_max']}")
print(f"Total unique monitoring sites (Areas): {groups['unique_areas']:,}")
print(f"Sites with <10 observations: {groups['areas_lt_10']:,} ({100*groups['areas_lt_10']/groups['unique_areas']:.1f}%)")
print(f"Max observations for a single site: {groups['max_area_obs']:,}")

### By Country:
| Country   |    Rows |   Percentage |
|-----------|---------|--------------|
| England   | 2129198 |        75.29 |
| USA       |  413814 |        14.63 |
| Ireland   |  235019 |         8.31 |
| China     |   45997 |         1.63 |
| Canada    |    3949 |         0.14 |

### By Waterbody Type:
| Waterbody Type   |    Rows |   Percentage |
|------------------|---------|--------------|
| River            | 1852579 |        65.51 |
| Effluent         |  601550 |        21.27 |
| Lake             |  153603 |         5.43 |
| Estuarine        |   49375 |         1.75 |
| Bay              |   45997 |         1.63 |
| Sea Water        |   32061 |         1.13 |
| Canal            |   28574 |         1.01 |
| Sewage           |   23777 |         0.84 |
| Marine           |   23162 |         0.82 |
| Drainage         |   10205 |         0.36 |
| Transitional     |    4275 |         0.15 |
| Coastal          |    2819 |         0.1  |

Temporal Span: 1940 – 2023
Total unique mo

## 5. Physical Plausibility Checks & Sensor Anomalies
Evaluate chemistry parameters against physical and physiological feasibility boundaries:
- **pH**: Neutral water is ~7.0; values <0 or >14 violate standard aqueous chemistry definitions (505 violations detected).
- **Temperature**: Environmental surface waters rarely exceed 40°C in temperate monitoring networks; temperatures >40°C (up to 98°C) indicate sensor failure or unit confusion (7,284 violations).
- **Dissolved Oxygen (DO)**: DO = 0 represents extreme anoxia or sensor zero-drift; DO values >20 mg/L represent super-saturation artefacts.
- **Negative Concentrations**: Concentrations cannot be physically negative; they often signify vendor-specific flags or below-detection-limit codes.

In [6]:
from src.stage1_analysis import plausibility_check, PLAUSIBLE_RANGES

plaus_df = plausibility_check(df)
print(tabulate(plaus_df, headers="keys", tablefmt="github", showindex=False))

| Ammonia (mg/l)   | Biochemical Oxygen Demand (mg/l)   | Dissolved Oxygen (mg/l)   | Orthophosphate (mg/l)   | pH (ph units)   | Temperature (cel)   | Nitrogen (mg/l)   | Nitrate (mg/l)   |
|------------------|------------------------------------|---------------------------|-------------------------|-----------------|---------------------|-------------------|------------------|
| below_min        | below_min                          | below_min                 | below_min               | below_min       | below_min           | below_min         | below_min        |
| above_max        | above_max                          | above_max                 | above_max               | above_max       | above_max           | above_max         | above_max        |
| exactly_zero     | exactly_zero                       | exactly_zero              | exactly_zero            | exactly_zero    | exactly_zero        | exactly_zero      | exactly_zero     |
| negative         | negative                

## 6. Target Variable & CCME Class Reverse Engineering
The continuous index `CCME_Values` ranges from 31.30 to 100.0.
A critical observation is the **ceiling effect**:
- **978,435 rows (34.6%) are capped at exactly 100.0** (censored regression target).
- The discrete classification label `CCME_WQI` is derived directly from `CCME_Values`.
Here we reverse-engineer the empirical threshold boundaries between continuous and categorical classes.

In [7]:
from src.stage1_analysis import reverse_engineer_ccme_classes

ccme_info = reverse_engineer_ccme_classes(df)

print(f"Rows capped at exactly 100.0: {ccme_info['n_capped_100']:,} ({ccme_info['pct_capped_100']:.2f}%)")
print("\n### Derived Class Thresholds:")
print(tabulate(ccme_info["summary_df"], headers="keys", tablefmt="github", showindex=False))

print("\n### Country vs CCME Class Distribution (% within Country):")
print(tabulate(ccme_info["country_class_pct"].round(2), headers="keys", tablefmt="github"))

Rows capped at exactly 100.0: 1,020,710 (36.09%)

### Derived Class Thresholds:
| Class     |   Min CCME |   Max CCME |   Mean CCME |     Count | % of Dataset   |
|-----------|------------|------------|-------------|-----------|----------------|
| Excellent |   100      |   100      |    100      | 1,020,710 | 36.09%         |
| Fair      |    65.0001 |    79.9983 |     73.2705 |   317,856 | 11.24%         |
| Good      |    80.0009 |    93.1957 |     89.1943 | 1,033,709 | 36.55%         |
| Marginal  |    45      |    65      |     53.4232 |   328,734 | 11.62%         |
| Poor      |    31.3041 |    45      |     42.4255 |   126,968 | 4.49%          |

### Country vs CCME Class Distribution (% within Country):
| Country   |   Excellent |   Fair |   Good |   Marginal |   Poor |
|-----------|-------------|--------|--------|------------|--------|
| Canada    |       35.48 |   5.93 |  52.19 |       4.81 |   1.6  |
| China     |       54.6  |   0.73 |  44.4  |       0.26 |   0    |
| Engla

## 7. Leakage Analysis: Mathematical Structure of CCME WQI
The Canadian Council of Ministers of the Environment (CCME) Water Quality Index is computed according to:
$$WQI = 100 - \frac{\sqrt{F_1^2 + F_2^2 + F_3^2}}{1.732}$$
Where:
- $F_1$ (**Scope**): Percentage of variables that do not meet guidelines.
- $F_2$ (**Frequency**): Percentage of individual tests that do not meet guidelines.
- $F_3$ (**Amplitude**): The amount by which failed tests exceed guidelines.

### Leakage Implication:
Because `CCME_Values` is an explicit, deterministic aggregation of guideline exceedances over the same water chemistry measurements present in the dataset:
- Standard ML models run the risk of simply approximating an algebraic formula instead of learning latent ecological properties.
- We quantify how well simple threshold-exceedance counts correlate with the target.

In [8]:
from src.stage1_analysis import leakage_analysis

leak = leakage_analysis(df)

print("Correlation between exceedance count and CCME_Values:")
print(f"  Pearson correlation:  {leak['correlation_exceedance_target']['pearson']:.4f}")
print(f"  Spearman correlation: {leak['correlation_exceedance_target']['spearman']:.4f}")
print(f"\nR² of linear regression predicting CCME_Values from simple exceedance count: {leak['r2_exceedance_alone']:.4f}")

print("\n### Mean CCME_Values by Exceedance Count:")
print(tabulate(leak["exceedance_stats"], headers="keys", tablefmt="github", showindex=False))

Correlation between exceedance count and CCME_Values:
  Pearson correlation:  -0.7695
  Spearman correlation: -0.8393

R² of linear regression predicting CCME_Values from simple exceedance count: 0.5922

### Mean CCME_Values by Exceedance Count:
|   Exceedance Count |      N |   Mean |   Std |   Median |
|--------------------|--------|--------|-------|----------|
|                  0 | 140975 |  98.98 |  2.58 |   100    |
|                  1 | 113803 |  93.76 |  8.48 |    93.19 |
|                  2 | 144280 |  80.52 | 13.92 |    86.6  |
|                  3 |  56998 |  67.46 | 16.76 |    70.65 |
|                  4 |  36532 |  56.73 | 13.79 |    52.35 |
|                  5 |   7063 |  50.94 | 12.23 |    45.34 |
|                  6 |    340 |  51.26 | 13.12 |    45.05 |
|                  7 |      9 |  40.21 |  3.66 |    40.56 |


## 8. Distribution Shifts & Shortcut Opportunities
We evaluate whether metadata features (Country, Waterbody Type, Era) could act as opportunistic shortcuts for machine learning models.

In [9]:
from src.stage1_analysis import distribution_shift_analysis

dist = distribution_shift_analysis(df)

print("### CCME_Values Distribution by Country:")
print(tabulate(dist["country_target"], headers="keys", tablefmt="github", showindex=False))

print("\n### CCME_Values Distribution by Waterbody Type:")
print(tabulate(dist["waterbody_target"], headers="keys", tablefmt="github", showindex=False))

### CCME_Values Distribution by Country:
| Country   |       N |   Mean |   Median |   Std |   % at 100 |
|-----------|---------|--------|----------|-------|------------|
| Canada    |    3949 |  90.22 |    93.12 | 12.36 |      35.48 |
| China     |   45997 |  96.51 |   100    |  4.62 |      54.6  |
| England   | 2129198 |  80.74 |    88.76 | 18.13 |      20.93 |
| Ireland   |  235019 |  98.11 |   100    |  6.41 |      84.24 |
| USA       |  413814 |  98.46 |   100    |  4.67 |      84.74 |

### CCME_Values Distribution by Waterbody Type:
| Waterbody Type   |       N |   Mean |   Median |   Std |
|------------------|---------|--------|----------|-------|
| Bay              |   45997 |  96.51 |   100    |  4.62 |
| Canal            |   28574 |  86.53 |    89.63 | 13.48 |
| Coastal          |    2819 |  93.7  |   100    | 16.15 |
| Drainage         |   10205 |  73.94 |    74.34 | 16.82 |
| Effluent         |  601550 |  66.56 |    67.59 | 19.11 |
| Estuarine        |   49375 |  85.12 |   

## 9. Stage 1 Key Takeaways & Transition to Stage 2
1. **Target Censoring**: ~35% of samples are capped at 100.0. The primary prediction task must be formulated as continuous regression on `CCME_Values`, with classification as a derived post-processing step.
2. **Formula Leakage**: Models must be evaluated carefully to distinguish genuine generalizable chemical relationships from trivial formula reconstruction.
3. **Plausibility Filters**: Stage 2 cleaning must remove physical impossibilities (pH outside [0, 14], extreme temperatures, negative concentrations) and isolate non-ambient sources (Effluent and Sewage).
4. **Metadata Boundaries**: `Country` and `Area` must serve as spatial splitting groups, not direct predictive features, to prevent shortcut learning.